# National Scale Financial Model — Zomato Platform (India)
Projects the cost savings from batching-optimized dispatch at national scale,
extrapolated from two calibrated city simulations (HD: Tier 2, HYD: Tier 1).

**This is a projection, not a claim.** The per-order saving rates are derived
from two simulated cities and scaled to national order volume using officially
sourced platform data from Eternal/Zomato Annual Reports FY24–FY26.

---
**Structure**
- **Section 1 — Constants & City Model Inputs**
- **Section 2 — Weighted Avg Per-Order Saving**
- **Section 3 — PRIMARY PROJECTION: Avg(P, Q)**
- **Section 4 — SUB-MODEL A: P–Q Range**
- **Section 5 — SUB-MODEL B: Aggressive Bound (P)**
- **Section 6 — SUB-MODEL C: Conservative Bound (Q)**

---
## Section 1 — Constants & City Model Inputs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#cccccc",
    "xtick.color":      "#cccccc",
    "ytick.color":      "#cccccc",
    "text.color":       "#ffffff",
    "grid.color":       "#333333",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
})

GREY  = "#555555"
GREEN = "#00c896"
CRORE = 1_00_00_000
DAYS_PER_YEAR = 365

# ── National platform constants — Eternal/Zomato Annual Reports FY24–FY26 ─────
# 3-year averages used throughout for consistency

# Total annual orders (millions)
ORDERS_FY24 = 753e6
ORDERS_FY25 = 853e6
ORDERS_FY26 = 979e6
AVG_ANNUAL_ORDERS    = (ORDERS_FY24 + ORDERS_FY25 + ORDERS_FY26) / 3
NATIONAL_DAILY_ORDERS = AVG_ANNUAL_ORDERS / DAYS_PER_YEAR

# Monthly active delivery partners (thousands)
PARTNERS_FY24 = 400e3
PARTNERS_FY25 = 473e3
PARTNERS_FY26 = 552e3
AVG_ACTIVE_PARTNERS  = (PARTNERS_FY24 + PARTNERS_FY25 + PARTNERS_FY26) / 3

# Restaurant partner count (thousands)
RESTAURANTS_FY24 = 247e3
RESTAURANTS_FY25 = 297e3
RESTAURANTS_FY26 = 330e3
AVG_RESTAURANTS      = (RESTAURANTS_FY24 + RESTAURANTS_FY25 + RESTAURANTS_FY26) / 3

print("── National Platform Constants (3-yr avg FY24–FY26) ─────────────")
print(f"  Avg annual orders        : {AVG_ANNUAL_ORDERS/1e6:.2f} million/year")
print(f"  National daily orders    : {NATIONAL_DAILY_ORDERS:,.0f} orders/day")
print(f"  Avg active partners/month: {AVG_ACTIVE_PARTNERS/1e3:.1f}K")
print(f"  Avg restaurant partners  : {AVG_RESTAURANTS/1e3:.1f}K")

In [ ]:
# ── City model outputs — locked from HD_Financial_Model & HYD_Financial_Model ─
# Source: HD_Financial_Model.ipynb — Avg(P,Q) primary model
HD_DAILY_ORDERS      = 1_515
HD_GREEDY_COST       = 38_352.74
HD_AVG_COST          = 37_090.705   # (P_COST + Q_COST) / 2
HD_P_COST            = 36_269.74
HD_Q_COST            = 37_911.67

HD_AVG_SAVING        = HD_GREEDY_COST - HD_AVG_COST
HD_P_SAVING          = HD_GREEDY_COST - HD_P_COST
HD_Q_SAVING          = HD_GREEDY_COST - HD_Q_COST

HD_AVG_PER_ORDER     = HD_AVG_SAVING / HD_DAILY_ORDERS
HD_P_PER_ORDER       = HD_P_SAVING   / HD_DAILY_ORDERS
HD_Q_PER_ORDER       = HD_Q_SAVING   / HD_DAILY_ORDERS

# Source: HYD_Financial_Model.ipynb — Avg(P,Q) primary model
HYD_DAILY_ORDERS     = 16_856
HYD_GREEDY_COST      = 279_466.99
HYD_AVG_COST         = 274_368.03   # (P_COST + Q_COST) / 2
HYD_P_COST           = 273_203.57
HYD_Q_COST           = 275_532.55

HYD_AVG_SAVING       = HYD_GREEDY_COST - HYD_AVG_COST
HYD_P_SAVING         = HYD_GREEDY_COST - HYD_P_COST
HYD_Q_SAVING         = HYD_GREEDY_COST - HYD_Q_COST

HYD_AVG_PER_ORDER    = HYD_AVG_SAVING / HYD_DAILY_ORDERS
HYD_P_PER_ORDER      = HYD_P_SAVING   / HYD_DAILY_ORDERS
HYD_Q_PER_ORDER      = HYD_Q_SAVING   / HYD_DAILY_ORDERS

print("── City Per-Order Savings ───────────────────────────────────────")
print(f"  HD  [Q]: ₹{HD_Q_PER_ORDER:.4f}  [Avg]: ₹{HD_AVG_PER_ORDER:.4f}  [P]: ₹{HD_P_PER_ORDER:.4f}")
print(f"  HYD [Q]: ₹{HYD_Q_PER_ORDER:.4f}  [Avg]: ₹{HYD_AVG_PER_ORDER:.4f}  [P]: ₹{HYD_P_PER_ORDER:.4f}")

---
## Section 2 — Weighted Avg Per-Order Saving
HD and HYD are weighted by their respective simulated order volumes.
Since HYD has ~11x more orders than HD, the blended rate sits closer to HYD's rate.
This is the most defensible approach given we have one Tier 1 and one Tier 2 data point.

In [ ]:
# ── Weighted per-order saving — P, Avg, Q ─────────────────────────────────────

TOTAL_SIM_ORDERS = HD_DAILY_ORDERS + HYD_DAILY_ORDERS

BLENDED_AVG_PER_ORDER = ((HD_DAILY_ORDERS * HD_AVG_PER_ORDER) +
                         (HYD_DAILY_ORDERS * HYD_AVG_PER_ORDER)) / TOTAL_SIM_ORDERS

BLENDED_P_PER_ORDER   = ((HD_DAILY_ORDERS * HD_P_PER_ORDER) +
                         (HYD_DAILY_ORDERS * HYD_P_PER_ORDER)) / TOTAL_SIM_ORDERS

BLENDED_Q_PER_ORDER   = ((HD_DAILY_ORDERS * HD_Q_PER_ORDER) +
                         (HYD_DAILY_ORDERS * HYD_Q_PER_ORDER)) / TOTAL_SIM_ORDERS

HD_WEIGHT  = HD_DAILY_ORDERS  / TOTAL_SIM_ORDERS * 100
HYD_WEIGHT = HYD_DAILY_ORDERS / TOTAL_SIM_ORDERS * 100

print("── Weighted Blending ────────────────────────────────────────────")
print(f"  HD  weight : {HD_WEIGHT:.1f}%  ({HD_DAILY_ORDERS:,} orders)")
print(f"  HYD weight : {HYD_WEIGHT:.1f}%  ({HYD_DAILY_ORDERS:,} orders)")
print()
print(f"  Blended per-order saving [Q]   : ₹{BLENDED_Q_PER_ORDER:.4f}")
print(f"  Blended per-order saving [Avg] : ₹{BLENDED_AVG_PER_ORDER:.4f}")
print(f"  Blended per-order saving [P]   : ₹{BLENDED_P_PER_ORDER:.4f}")

In [ ]:
# ── Chart: Per-order saving across cities and blended rate ────────────────────

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle("National Model — Per-Order Saving Rate by City & Blended",
             fontsize=13, fontweight="bold", color="white", y=1.02)

labels   = ["HD\n(Tier 2)", "HYD\n(Tier 1)", "Blended\n(Weighted Avg)"]
q_vals   = [HD_Q_PER_ORDER,   HYD_Q_PER_ORDER,   BLENDED_Q_PER_ORDER]
avg_vals = [HD_AVG_PER_ORDER, HYD_AVG_PER_ORDER, BLENDED_AVG_PER_ORDER]
p_vals   = [HD_P_PER_ORDER,   HYD_P_PER_ORDER,   BLENDED_P_PER_ORDER]

x     = np.arange(len(labels))
width = 0.25

ax.bar(x - width, q_vals,   width, label="Q (Conservative)", color="#4488ff", zorder=3)
ax.bar(x,         avg_vals, width, label="Avg P+Q (Primary)", color=GREEN,    zorder=3)
ax.bar(x + width, p_vals,   width, label="P (Aggressive)",   color="#ff9900", zorder=3)

for i, (q, avg, p) in enumerate(zip(q_vals, avg_vals, p_vals)):
    ax.text(i - width, q   + 0.005, f"₹{q:.3f}",   ha="center", fontsize=8, color="#4488ff")
    ax.text(i,         avg + 0.005, f"₹{avg:.3f}", ha="center", fontsize=8, color=GREEN)
    ax.text(i + width, p   + 0.005, f"₹{p:.3f}",   ha="center", fontsize=8, color="#ff9900")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("₹ saved per order")
ax.set_title("HD (Tier 2) has higher per-order saving due to lower order density vs HYD (Tier 1)",
             fontsize=9, color="#aaaaaa", pad=6)
ax.legend(fontsize=9, facecolor="#1a1a1a", edgecolor="#444")
ax.grid(axis="y", zorder=0)

plt.tight_layout()
plt.savefig("per_order_saving.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → per_order_saving.png")

---
## Section 3 — PRIMARY PROJECTION: Avg(P, Q)
Central estimate using the blended weighted-average per-order saving applied
to the 3-year average national daily order volume.

In [ ]:
# ── Primary projection — Avg(P, Q) ───────────────────────────────────────────

nat_daily_saving_avg  = NATIONAL_DAILY_ORDERS * BLENDED_AVG_PER_ORDER
nat_annual_saving_avg = nat_daily_saving_avg * DAYS_PER_YEAR
nat_annual_cr_avg     = nat_annual_saving_avg / CRORE

print("══ PRIMARY PROJECTION — Avg(P, Q) ══════════════════════════════")
print(f"  National daily orders          : {NATIONAL_DAILY_ORDERS:,.0f}")
print(f"  Blended per-order saving [Avg] : ₹{BLENDED_AVG_PER_ORDER:.4f}")
print()
print(f"  Projected daily saving         : ₹{nat_daily_saving_avg:,.0f}")
print(f"  Projected annual saving        : ₹{nat_annual_saving_avg:,.0f}")
print(f"  Projected annual saving        : ₹{nat_annual_cr_avg:.4f} crore/year")

In [ ]:
# ── Chart 1: Annual projection — order volume trend + saving projection ───────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("National Financial Projection — PRIMARY: Avg(P, Q)",
             fontsize=13, fontweight="bold", color="white", y=1.02)

# — Panel 1: Order volume trend FY24–FY26 + projection ────────────────────────
ax = axes[0]
years  = ["FY24", "FY25", "FY26", "3-yr Avg\n(Used)"]
orders = [753, 853, 979, AVG_ANNUAL_ORDERS/1e6]
colors = [GREY, GREY, GREY, GREEN]
bars   = ax.bar(years, orders, color=colors, width=0.5, zorder=3)
ax.set_title("Annual Order Volume (million)", fontsize=11, pad=10)
ax.set_ylabel("Orders (million)")
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, orders):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{val:.0f}M", ha="center", va="bottom", fontsize=9, color="white")
ax.set_ylim(0, 1100)
ax.axhline(y=AVG_ANNUAL_ORDERS/1e6, color=GREEN, linestyle="--", alpha=0.4, zorder=2)

# — Panel 2: Annual saving projection gauge ───────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Annual Cost Saving Projection", fontsize=11, pad=10)
ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.02), 0.9, 0.94,
             boxstyle="round,pad=0.02", linewidth=2,
             edgecolor=GREEN, facecolor="#1a1a1a"))
ax.text(0.5, 0.78, f"₹{nat_annual_cr_avg:.3f}",
        ha="center", va="center", fontsize=32, fontweight="bold", color=GREEN)
ax.text(0.5, 0.60, "crore / year",
        ha="center", va="center", fontsize=14, color="#cccccc")
ax.text(0.5, 0.44, f"Daily saving: ₹{nat_daily_saving_avg:,.0f}",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.30, f"Based on {NATIONAL_DAILY_ORDERS/1e6:.2f}M orders/day",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.16, f"Blended rate: ₹{BLENDED_AVG_PER_ORDER:.4f}/order",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.05, "★ Projection based on HD (Tier 2) + HYD (Tier 1) simulations",
        ha="center", va="center", fontsize=8, color="#666666")

plt.tight_layout()
plt.savefig("primary_projection.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → primary_projection.png")

In [ ]:
# ── Chart 2: Partner pool & platform scale context ────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("National Platform Scale — Partner Pool & Restaurant Network (FY24–FY26)",
             fontsize=12, fontweight="bold", color="white", y=1.02)

fy_labels = ["FY24", "FY25", "FY26", "3-yr Avg"]

# — Panel 1: Active delivery partners ─────────────────────────────────────────
ax = axes[0]
partner_vals = [400, 473, 552, AVG_ACTIVE_PARTNERS/1e3]
bar_colors   = [GREY, GREY, GREY, GREEN]
bars = ax.bar(fy_labels, partner_vals, color=bar_colors, width=0.5, zorder=3)
ax.set_title("Monthly Active Delivery Partners (K)", fontsize=11, pad=10)
ax.set_ylabel("Partners (thousands)")
ax.grid(axis="y", zorder=0)
ax.set_ylim(0, 650)
for bar, val in zip(bars, partner_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{val:.0f}K", ha="center", va="bottom", fontsize=9, color="white")
ax.axhline(y=AVG_ACTIVE_PARTNERS/1e3, color=GREEN, linestyle="--", alpha=0.4, zorder=2)

# — Panel 2: Restaurant partners ──────────────────────────────────────────────
ax = axes[1]
rest_vals  = [247, 297, 330, AVG_RESTAURANTS/1e3]
bars = ax.bar(fy_labels, rest_vals, color=bar_colors, width=0.5, zorder=3)
ax.set_title("Restaurant Partner Count (K)", fontsize=11, pad=10)
ax.set_ylabel("Restaurants (thousands)")
ax.grid(axis="y", zorder=0)
ax.set_ylim(0, 400)
for bar, val in zip(bars, rest_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f"{val:.0f}K", ha="center", va="bottom", fontsize=9, color="white")
ax.axhline(y=AVG_RESTAURANTS/1e3, color=GREEN, linestyle="--", alpha=0.4, zorder=2)

plt.tight_layout()
plt.savefig("platform_scale.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → platform_scale.png")

---
## Section 4 — SUB-MODEL A: P–Q Range
Full uncertainty band of the national projection — conservative (Q) to aggressive (P).

In [ ]:
# ── P–Q Range ─────────────────────────────────────────────────────────────────

nat_daily_saving_P  = NATIONAL_DAILY_ORDERS * BLENDED_P_PER_ORDER
nat_daily_saving_Q  = NATIONAL_DAILY_ORDERS * BLENDED_Q_PER_ORDER
nat_annual_cr_P     = (nat_daily_saving_P * DAYS_PER_YEAR) / CRORE
nat_annual_cr_Q     = (nat_daily_saving_Q * DAYS_PER_YEAR) / CRORE

print("══ SUB-MODEL A — P–Q Range ══════════════════════════════════════")
print(f"  Blended per-order saving [Q]   : ₹{BLENDED_Q_PER_ORDER:.4f}")
print(f"  Blended per-order saving [P]   : ₹{BLENDED_P_PER_ORDER:.4f}")
print()
print(f"  Projected daily saving  [Q–P]  : ₹{nat_daily_saving_Q:,.0f} – ₹{nat_daily_saving_P:,.0f}")
print(f"  Projected annual saving [Q–P]  : ₹{nat_annual_cr_Q:.4f} cr – ₹{nat_annual_cr_P:.4f} cr")

In [ ]:
# ── Chart: P–Q–Avg range band ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("National Projection — P–Q Savings Range",
             fontsize=13, fontweight="bold", color="white", y=1.02)

scenarios   = ["Q\n(Conservative)", "Avg\n(Primary)", "P\n(Aggressive)"]
daily_vals  = [nat_daily_saving_Q,  nat_daily_saving_avg,  nat_daily_saving_P]
annual_cr   = [nat_annual_cr_Q,     nat_annual_cr_avg,     nat_annual_cr_P]
bar_colors  = ["#4488ff", GREEN, "#ff9900"]

# — Panel 1: Daily saving ─────────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(scenarios, daily_vals, color=bar_colors, width=0.4, zorder=3)
ax.set_title("Projected Daily Saving (₹)", fontsize=11, pad=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"₹{x:,.0f}"))
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, daily_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(daily_vals)*0.01,
            f"₹{val:,.0f}", ha="center", va="bottom", fontsize=9, color="white")

# — Panel 2: Annual saving ────────────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(scenarios, annual_cr, color=bar_colors, width=0.4, zorder=3)
ax.set_title("Projected Annual Saving (₹ crore)", fontsize=11, pad=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"₹{x:.3f} cr"))
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, annual_cr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(annual_cr)*0.01,
            f"₹{val:.4f} cr", ha="center", va="bottom", fontsize=9, color="white")
ax.set_title("Q = Lower Bound  |  Avg = Primary  |  P = Upper Bound",
             fontsize=9, color="#aaaaaa", pad=6)

plt.tight_layout()
plt.savefig("range_comparison.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → range_comparison.png")

---
## Section 5 — SUB-MODEL B: Aggressive Bound (P)
Upper bound of the national projection — maximum batching efficiency applied nationally.

In [ ]:
print("══ SUB-MODEL B — Aggressive Bound (P) ══════════════════════════")
print(f"  Blended per-order saving [P]   : ₹{BLENDED_P_PER_ORDER:.4f}")
print(f"  Projected daily saving         : ₹{nat_daily_saving_P:,.0f}")
print(f"  Projected annual saving        : ₹{nat_annual_cr_P:.4f} crore/year")

In [ ]:
# ── Chart: Sub-Model B — Aggressive Bound (P) ────────────────────────────────

P_COLOR = "#ff9900"
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sub-Model B — Aggressive Bound (P)  |  National Upper Bound Projection",
             fontsize=12, fontweight="bold", color="white", y=1.02)

# — Panel 1: City vs blended P rate ───────────────────────────────────────────
ax = axes[0]
cities = ["HD\n(Tier 2)", "HYD\n(Tier 1)", "Blended\n(National)"]
p_rates = [HD_P_PER_ORDER, HYD_P_PER_ORDER, BLENDED_P_PER_ORDER]
bars = ax.bar(cities, p_rates, color=[GREY, GREY, P_COLOR], width=0.5, zorder=3)
ax.set_title("Per-Order Saving Rate [P] by City", fontsize=11, pad=10)
ax.set_ylabel("₹ saved per order")
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, p_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"₹{val:.4f}", ha="center", va="bottom", fontsize=9, color="white")

# — Panel 2: Annual saving gauge ──────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("National Annual Saving [P]", fontsize=11, pad=10)
ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.02), 0.9, 0.94,
             boxstyle="round,pad=0.02", linewidth=2,
             edgecolor=P_COLOR, facecolor="#1a1a1a"))
ax.text(0.5, 0.75, f"₹{nat_annual_cr_P:.4f}",
        ha="center", va="center", fontsize=30, fontweight="bold", color=P_COLOR)
ax.text(0.5, 0.57, "crore / year",
        ha="center", va="center", fontsize=13, color="#cccccc")
ax.text(0.5, 0.40, f"Daily saving: ₹{nat_daily_saving_P:,.0f}",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.26, f"Blended rate: ₹{BLENDED_P_PER_ORDER:.4f}/order",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.13, f"Avg active partners: {AVG_ACTIVE_PARTNERS/1e3:.0f}K pool",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.04, "★ Upper bound — aggressive batching scenario",
        ha="center", va="center", fontsize=8, color="#666666")

plt.tight_layout()
plt.savefig("submodel_p.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → submodel_p.png")

---
## Section 6 — SUB-MODEL C: Conservative Bound (Q)
Lower bound — minimum guaranteed savings if smart batching is deployed nationally.

In [ ]:
print("══ SUB-MODEL C — Conservative Bound (Q) ════════════════════════")
print(f"  Blended per-order saving [Q]   : ₹{BLENDED_Q_PER_ORDER:.4f}")
print(f"  Projected daily saving         : ₹{nat_daily_saving_Q:,.0f}")
print(f"  Projected annual saving        : ₹{nat_annual_cr_Q:.4f} crore/year")

In [ ]:
# ── Chart: Sub-Model C — Conservative Bound (Q) ──────────────────────────────

Q_COLOR = "#4488ff"
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sub-Model C — Conservative Bound (Q)  |  National Lower Bound Projection",
             fontsize=12, fontweight="bold", color="white", y=1.02)

# — Panel 1: City vs blended Q rate ───────────────────────────────────────────
ax = axes[0]
q_rates = [HD_Q_PER_ORDER, HYD_Q_PER_ORDER, BLENDED_Q_PER_ORDER]
bars = ax.bar(cities, q_rates, color=[GREY, GREY, Q_COLOR], width=0.5, zorder=3)
ax.set_title("Per-Order Saving Rate [Q] by City", fontsize=11, pad=10)
ax.set_ylabel("₹ saved per order")
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, q_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"₹{val:.4f}", ha="center", va="bottom", fontsize=9, color="white")

# — Panel 2: Annual saving gauge ──────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("National Annual Saving [Q]", fontsize=11, pad=10)
ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.02), 0.9, 0.94,
             boxstyle="round,pad=0.02", linewidth=2,
             edgecolor=Q_COLOR, facecolor="#1a1a1a"))
ax.text(0.5, 0.75, f"₹{nat_annual_cr_Q:.4f}",
        ha="center", va="center", fontsize=30, fontweight="bold", color=Q_COLOR)
ax.text(0.5, 0.57, "crore / year",
        ha="center", va="center", fontsize=13, color="#cccccc")
ax.text(0.5, 0.40, f"Daily saving: ₹{nat_daily_saving_Q:,.0f}",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.26, f"Blended rate: ₹{BLENDED_Q_PER_ORDER:.4f}/order",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.13, f"Avg active partners: {AVG_ACTIVE_PARTNERS/1e3:.0f}K pool",
        ha="center", va="center", fontsize=10, color="#aaaaaa")
ax.text(0.5, 0.04, "★ Lower bound — conservative batching scenario",
        ha="center", va="center", fontsize=8, color="#666666")

plt.tight_layout()
plt.savefig("submodel_q.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → submodel_q.png")

In [ ]:
# ── Master summary table ──────────────────────────────────────────────────────

summary = pd.DataFrame({
    "Metric": [
        "National daily orders (3-yr avg)",
        "Avg active partners/month (3-yr avg)",
        "Avg restaurant partners (3-yr avg)",
        "Blended per-order saving [Q]",
        "Blended per-order saving [Avg]",
        "Blended per-order saving [P]",
        "Projected daily saving [Q]",
        "Projected daily saving [Avg]",
        "Projected daily saving [P]",
        "Projected annual saving [Q] (₹ crore)",
        "Projected annual saving [Avg] (₹ crore)",
        "Projected annual saving [P] (₹ crore)",
    ],
    "Value": [
        f"{NATIONAL_DAILY_ORDERS:,.0f} orders/day",
        f"{AVG_ACTIVE_PARTNERS/1e3:.1f}K partners",
        f"{AVG_RESTAURANTS/1e3:.1f}K restaurants",
        f"₹{BLENDED_Q_PER_ORDER:.4f}/order",
        f"₹{BLENDED_AVG_PER_ORDER:.4f}/order",
        f"₹{BLENDED_P_PER_ORDER:.4f}/order",
        f"₹{nat_daily_saving_Q:,.0f}",
        f"₹{nat_daily_saving_avg:,.0f}",
        f"₹{nat_daily_saving_P:,.0f}",
        f"₹{nat_annual_cr_Q:.4f} cr",
        f"₹{nat_annual_cr_avg:.4f} cr",
        f"₹{nat_annual_cr_P:.4f} cr",
    ]
})

print("══ National Financial Model — Master Summary Table ═════════════")
print(summary.to_string(index=False))
print()
print("  NOTE: This is a projection extrapolated from HD (Tier 2) and HYD (Tier 1)")
print("        simulations. Not a claim. Presented as a [Q, Avg, P] sensitivity range.")